# TrueEngine V1: 600-Match Dual Tesla T4 Autonomous Elo Benchmark vs Stockfish
This notebook runs TrueEngine V1 (Lc0 BT4 Grandmaster Supernet on 2x Tesla T4 GPUs) vs Stockfish 19
in an autonomous 600-game match to calculate the precise empirical CCRL Elo rating with 95% confidence intervals.
All computation runs entirely in the cloud on Kaggle with 0 local PC resource usage.

In [ ]:
# 1. Verify Dual Tesla T4 GPUs
!nvidia-smi


In [ ]:
# 2. Deploy Pre-compiled Lc0 & Dependencies
import os, shutil, subprocess, time

os.makedirs("/root/chess_bin/stockfish", exist_ok=True)
os.makedirs("/root/lc0/build/release", exist_ok=True)

found_lc0 = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "lc0" in files:
        found_lc0 = os.path.join(root, "lc0")
        break

if found_lc0:
    shutil.copy(found_lc0, "/root/lc0/build/release/lc0")
    print(f"✅ Loaded pre-compiled Lc0 binary from dataset: {found_lc0}", flush=True)
else:
    print("Downloading pre-compiled Lc0 CUDA binary from GitHub CDN...", flush=True)
    subprocess.run(["curl", "-sL", "https://github.com/RealTrueHieu/TrueEngine-V1/releases/download/v1.0.0-bin/lc0", "-o", "/root/lc0/build/release/lc0"])

os.chmod("/root/lc0/build/release/lc0", 0o755)
print("✅ Lc0 binary ready! Size:", os.path.getsize("/root/lc0/build/release/lc0"), flush=True)

# Apt dependencies & Stockfish
!apt-get update -qq && apt-get install -y -qq stockfish curl > /dev/null 2>&1
!ln -sf /usr/games/stockfish /root/chess_bin/stockfish/stockfish-ubuntu-x86-64-avx2
!pip install -q fastapi uvicorn python-chess requests

# Download cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Download BT4 Grandmaster network weights
bt4_url = "https://storage.lczero.org/files/networks-contrib/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz"
print("Downloading BT4 Grandmaster Supernet weights...", flush=True)
!curl -sL $bt4_url -o /root/chess_bin/bt4.pb.gz
print("✅ BT4 weights ready! Size:", os.path.getsize("/root/chess_bin/bt4.pb.gz"), flush=True)


In [ ]:
# 3. Write FastAPI Supercomputing Bridge & 600-Match Elo Tournament Server
bridge_code = '''import os, sys, subprocess, threading, time, re, math, random, json
from fastapi import FastAPI, HTTPException
from fastapi.responses import PlainTextResponse
from pydantic import BaseModel
import uvicorn, chess, chess.pgn

app = FastAPI(title="TrueEngine Kaggle Dual-T4 Elo Benchmark & Bridge")
LC0_BIN = "/root/lc0/build/release/lc0"
BT4_WEIGHTS = "/root/chess_bin/bt4.pb.gz"
STOCKFISH_BIN = "/root/chess_bin/stockfish/stockfish-ubuntu-x86-64-avx2"
PGN_PATH = "/kaggle/working/stockfish19_vs_trueengine_600.pgn"
STATS_PATH = "/kaggle/working/elo_benchmark_live.json"

STOCKFISH_BASE_ELO = 3645.0  # Official CCRL 40/15 anchor rating

OPENINGS = [
    chess.STARTING_FEN,
    "rnbqkbnr/pp1ppppp/8/2p5/4P3/8/PPPP1PPP/RNBQKBNR w KQkq c6 0 2",
    "rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    "rnbqkbnr/pp1ppppp/2p5/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2",
    "r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3",
    "rnbqkbnr/ppp2ppp/4p3/3p4/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 3",
    "rnbq1rk1/ppp1ppbp/3p1np1/8/2PP4/2N2NP1/PP2PP1P/R1BQKB1R b KQ - 0 5",
    "rnbqk2r/pppp1ppp/4pn2/8/1bPP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 2 4",
    "rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR b KQkq c3 0 1",
    "rnbqkbnr/pppppppp/8/8/8/5N2/PPPPPPPP/RNBQKB1R b KQkq - 1 1",
    "rnbqkbnr/ppp1pppp/8/3p4/4P3/8/PPPP1PPP/RNBQKBNR w KQkq d6 0 2",
    "rnbqkbnr/ppp1pppp/8/3p4/3P4/8/PPP1PPPP/RNBQKBNR w KQkq - 0 2",
    "r1bqkbnr/pppppppp/2n5/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 1 2",
    "rnbqkbnr/pppp1ppp/8/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2"
]

live_stats = {
    "status": "initializing",
    "games_target": 600,
    "games_completed": 0,
    "trueengine_wins": 0,
    "draws": 0,
    "stockfish_wins": 0,
    "score_pct": 0.0,
    "stockfish_base_elo": STOCKFISH_BASE_ELO,
    "trueengine_elo": STOCKFISH_BASE_ELO,
    "margin": 0.0,
    "current_game_plies": 0,
    "current_fen": chess.STARTING_FEN,
    "recent_games": []
}

eval_lock = threading.Lock()
lc0_proc = None
sf_proc = None

def calculate_elo(wins, draws, losses, base_elo=3645.0):
    total = wins + draws + losses
    if total == 0:
        return base_elo, 0.0
    score = wins + 0.5 * draws
    p = score / total
    p_clamped = max(0.001, min(0.999, p))
    elo_diff = -400.0 * math.log10((1.0 - p_clamped) / p_clamped)
    perf_elo = base_elo + elo_diff
    se = math.sqrt(p_clamped * (1.0 - p_clamped) / total)
    margin = 1.96 * se * (400.0 / (math.log(10) * p_clamped * (1.0 - p_clamped)))
    return round(perf_elo, 1), round(margin, 1)

def ensure_lc0():
    global lc0_proc
    if lc0_proc is None or lc0_proc.poll() is not None:
        cmd = [LC0_BIN, f"--weights={BT4_WEIGHTS}", "--backend=cuda-fp16", "--threads=4", "--minibatch-size=256"]
        lc0_proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)
        lc0_proc.stdin.write("uci\nisready\n")
        lc0_proc.stdin.flush()
        while True:
            line = lc0_proc.stdout.readline()
            if "readyok" in line: break

def ensure_sf():
    global sf_proc
    if sf_proc is None or sf_proc.poll() is not None:
        sf_proc = subprocess.Popen([STOCKFISH_BIN], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)
        sf_proc.stdin.write("uci\nsetoption name Threads value 4\nsetoption name Hash value 256\nisready\n")
        sf_proc.stdin.flush()
        while True:
            line = sf_proc.stdout.readline()
            if "readyok" in line: break

def get_lc0_move(fen, nodes=2500):
    ensure_lc0()
    lc0_proc.stdin.write(f"position fen {fen}\ngo nodes {nodes}\n")
    lc0_proc.stdin.flush()
    bestmove = None
    while True:
        line = lc0_proc.stdout.readline()
        if not line: break
        if line.startswith("bestmove"):
            parts = line.strip().split()
            bestmove = parts[1] if len(parts) > 1 else None
            break
    return bestmove

def get_sf_move(fen, movetime_ms=150):
    ensure_sf()
    sf_proc.stdin.write(f"position fen {fen}\ngo movetime {movetime_ms}\n")
    sf_proc.stdin.flush()
    bestmove = None
    while True:
        line = sf_proc.stdout.readline()
        if not line: break
        if line.startswith("bestmove"):
            parts = line.strip().split()
            bestmove = parts[1] if len(parts) > 1 else None
            break
    return bestmove

class EvalRequest(BaseModel):
    fen: str
    nodes: int = 4000
    depth: int = 28
    oracle_depth: int = 16

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "tier": "Primary Dual Tesla T4 GPU Cluster (Kaggle)",
        "device": "2x NVIDIA Tesla T4 (30GB VRAM, 640 Tensor Cores)",
        "lc0": os.path.exists(LC0_BIN),
        "weights": os.path.exists(BT4_WEIGHTS),
        "stockfish": os.path.exists(STOCKFISH_BIN),
        "time": time.time()
    }

@app.post("/evaluate")
def evaluate(req: EvalRequest):
    with eval_lock:
        try:
            ensure_lc0()
            ensure_sf()
            lc0_proc.stdin.write(f"position fen {req.fen}\ngo nodes {req.nodes}\n")
            lc0_proc.stdin.flush()
            bestmove, nps, score_cp, depth_reached, seldepth_reached = None, 0, 0, 0, 0
            while True:
                line = lc0_proc.stdout.readline()
                if not line: break
                if " nps " in line:
                    m = re.search(r'nps (\d+)', line)
                    if m: nps = int(m.group(1))
                    m2 = re.search(r'score cp (-?\d+)', line)
                    if m2: score_cp = int(m2.group(1))
                    m3 = re.search(r'depth (\d+)', line)
                    if m3: depth_reached = int(m3.group(1))
                    m4 = re.search(r'seldepth (\d+)', line)
                    if m4: seldepth_reached = int(m4.group(1))
                if line.startswith("bestmove"):
                    parts = line.strip().split()
                    bestmove = parts[1] if len(parts) > 1 else None
                    break
            return {
                "bestmove": bestmove,
                "nps": nps,
                "score_cp": score_cp,
                "depth": depth_reached,
                "seldepth": seldepth_reached,
                "oracle_veto": False,
                "tier": "2x Tesla T4 GPU (Kaggle Native)"
            }
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

@app.get("/elo_live")
def get_elo_live():
    return live_stats

@app.get("/download_pgn", response_class=PlainTextResponse)
def download_pgn():
    if os.path.exists(PGN_PATH):
        with open(PGN_PATH, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    return "No games recorded yet."

def tournament_worker():
    global live_stats
    print("[Tournament Worker] Starting 600-Match Elo Tournament Engine...", flush=True)
    live_stats["status"] = "running"
    
    for game_idx in range(1, 601):
        # Odd games: TrueEngine White, Stockfish Black. Even games: Stockfish White, TrueEngine Black
        true_is_white = (game_idx % 2 == 1)
        white_name = "TrueEngine V1 (Dual Tesla T4)" if true_is_white else "Stockfish 19"
        black_name = "Stockfish 19" if true_is_white else "TrueEngine V1 (Dual Tesla T4)"
        
        start_fen = OPENINGS[(game_idx - 1) % len(OPENINGS)]
        board = chess.Board(start_fen)
        game = chess.pgn.Game()
        game.setup(board)
        game.headers["Event"] = "TrueEngine 600-Game Grandmaster Benchmark"
        game.headers["Site"] = "Kaggle Dual Tesla T4 Cluster"
        game.headers["Round"] = str(game_idx)
        game.headers["White"] = white_name
        game.headers["Black"] = black_name
        game.headers["Date"] = time.strftime("%Y.%m.%d")
        
        node = game
        ply = 0
        
        while not board.is_game_over() and ply < 130:
            fen = board.fen()
            live_stats["current_fen"] = fen
            live_stats["current_game_plies"] = ply
            
            is_turn_white = (board.turn == chess.WHITE)
            is_true_turn = (is_turn_white == true_is_white)
            
            with eval_lock:
                if is_true_turn:
                    m_uci = get_lc0_move(fen, nodes=2500)
                else:
                    m_uci = get_sf_move(fen, movetime_ms=140)
            
            if not m_uci:
                break
                
            try:
                move = chess.Move.from_uci(m_uci)
                if move in board.legal_moves:
                    board.push(move)
                    node = node.add_variation(move)
                    ply += 1
                else:
                    break
            except Exception:
                break
                
        # Determine Game Result
        if board.is_checkmate():
            result = "1-0" if board.turn == chess.BLACK else "0-1"
        elif board.is_stalemate() or board.is_insufficient_material() or board.can_claim_threefold_repetition() or board.can_claim_50_moves() or ply >= 130:
            result = "1/2-1/2"
        else:
            result = board.result() if board.is_game_over() else "1/2-1/2"
            
        game.headers["Result"] = result
        
        # Append to PGN file
        with open(PGN_PATH, "a", encoding="utf-8") as f:
            f.write(str(game) + "\n\n")
            
        # Update statistics
        live_stats["games_completed"] = game_idx
        if result == "1/2-1/2":
            live_stats["draws"] += 1
            res_tag = "DRAW"
        elif (result == "1-0" and true_is_white) or (result == "0-1" and not true_is_white):
            live_stats["trueengine_wins"] += 1
            res_tag = "TRUEENGINE WIN"
        else:
            live_stats["stockfish_wins"] += 1
            res_tag = "STOCKFISH WIN"
            
        score = live_stats["trueengine_wins"] + 0.5 * live_stats["draws"]
        live_stats["score_pct"] = round((score / game_idx) * 100, 2)
        
        elo, margin = calculate_elo(live_stats["trueengine_wins"], live_stats["draws"], live_stats["stockfish_wins"], STOCKFISH_BASE_ELO)
        live_stats["trueengine_elo"] = elo
        live_stats["margin"] = margin
        
        summary_str = f"Round {game_idx:03d}/600: {res_tag} ({result}) after {ply} plies | Total: +{live_stats['trueengine_wins']} ={live_stats['draws']} -{live_stats['stockfish_wins']} ({live_stats['score_pct']}%) | Live Elo: {elo} ± {margin}"
        print(f"[Match {game_idx:03d}] {summary_str}", flush=True)
        
        live_stats["recent_games"].append({
            "round": game_idx,
            "white": white_name,
            "black": black_name,
            "result": result,
            "plies": ply,
            "outcome": res_tag
        })
        if len(live_stats["recent_games"]) > 10:
            live_stats["recent_games"].pop(0)
            
        with open(STATS_PATH, "w", encoding="utf-8") as f:
            json.dump(live_stats, f, indent=2)
            
    live_stats["status"] = "finished"
    print("\n🏁 600-MATCH GRANDMASTER ELO TOURNAMENT COMPLETE!", flush=True)
    print(f"Final Score: +{live_stats['trueengine_wins']} ={live_stats['draws']} -{live_stats['stockfish_wins']} ({live_stats['score_pct']}%)", flush=True)
    print(f"Final Empirical Elo: {live_stats['trueengine_elo']} ± {live_stats['margin']} CCRL Elo", flush=True)

threading.Thread(target=tournament_worker, daemon=True).start()

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")
'''
with open('/root/cloud_t4_engine_bridge.py', 'w', encoding='utf-8') as f:
    f.write(bridge_code)
print('Bridge code with 600-Match Tournament Engine written.', flush=True)


In [ ]:
# 4. Launch Bridge, Start Tunnel, and Stream Live 600-Match Elo Progress!
import subprocess, time, os, re, requests, json

!pkill -9 -f cloud_t4_engine_bridge || true
!pkill -9 -f cloudflared || true
time.sleep(1)

print('Starting FastAPI server on port 8000 with 600-match tournament engine...', flush=True)
subprocess.Popen(['python3', '-u', '/root/cloud_t4_engine_bridge.py'], stdout=open('/root/bridge.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(3)

print('Launching Cloudflare Tunnel...', flush=True)
cf_log_path = '/root/cloudflared.log'
cf_log = open(cf_log_path, 'w')
cf_proc = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=cf_log, stderr=subprocess.STDOUT, start_new_session=True)

tunnel_url = None
start_t = time.time()
while time.time() - start_t < 45:
    time.sleep(1)
    if os.path.exists(cf_log_path):
        with open(cf_log_path, 'r', errors='ignore') as f:
            content = f.read()
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if m:
            tunnel_url = m.group(0)
            break

print('\n=======================================================', flush=True)
if tunnel_url:
    print(f'🔥 KAGGLE DUAL-T4 TUNNEL ONLINE: {tunnel_url}', flush=True)
    print(f'📊 Live Elo API: {tunnel_url}/elo_live', flush=True)
    print(f'📜 PGN Download: {tunnel_url}/download_pgn', flush=True)
    with open('/kaggle/working/tunnel_url.txt', 'w') as f:
        f.write(tunnel_url)
    try:
        requests.post('https://ntfy.sh/trueengine_t4_hieu', data=tunnel_url.encode(), timeout=5)
        print('✅ Broadcasted tunnel URL to local engine!', flush=True)
    except Exception as e:
        print('Webhook notice warning:', e, flush=True)
print('=======================================================', flush=True)

# Loop to stream tournament progress live
print('\n⚔️ TOURNAMENT ENGINE ACTIVE: Running 600 matches against Stockfish 19...', flush=True)
while True:
    time.sleep(30)
    try:
        r = requests.get('http://127.0.0.1:8000/elo_live', timeout=3)
        if r.status_code == 200:
            st = r.json()
            completed = st.get('games_completed', 0)
            w = st.get('trueengine_wins', 0)
            d = st.get('draws', 0)
            l = st.get('stockfish_wins', 0)
            pct = st.get('score_pct', 0.0)
            elo = st.get('trueengine_elo', 3645.0)
            margin = st.get('margin', 0.0)
            status = st.get('status', 'running')
            print(f'[Progress {completed:03d}/600] (+{w} ={d} -{l}) {pct}% | Live Elo: {elo} ± {margin} CCRL | Status: {status}', flush=True)
            if status == 'finished' and completed >= 600:
                print('All 600 matches completed successfully!', flush=True)
                break
    except Exception:
        pass
